# RF Cavity — EigenspaceOperator (NEO) eğitimi

**Saf model tahmini:** ağ, mesh üzerinde 16 baz fonksiyonu üretir → 16×16 Rayleigh–Ritz → 3 mod (alan + frekans). FEM çözücü kullanılmaz (`REFINE = 0`).

**Akış:** repo → ayarlar → kurulum → veri üretimi (gmsh + P2 FEM etiketleri) → 6 modlu PKL → eğitim (Drive'a checkpoint, kesintide kaldığı yerden devam) → test değerlendirmesi (topoloji × özdeğer aralığı tablosu) → görseller.

**Önce** *Runtime → Change runtime type → GPU* seçin. İlk kez `MODE = "smoke"` ile birkaç dakikada tüm akışı deneyin, sonra `MODE = "full"`.

In [ ]:
# ── 1. Repo ─────────────────────────────────────────────────────────────
import os
BRANCH = "claude/neo-eigenspace"
REPO = "/content/rf_cavity_neural_operator"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/KorayGokceler/rf_cavity_neural_operator.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

In [ ]:
# ── 2. Ayarlar ──────────────────────────────────────────────────────────
MODE = "smoke"            # "smoke": ~40 geometri, 3 epoch (akışı dener) | "full": gerçek eğitim
USE_DRIVE = True          # veri + checkpoint Google Drive'da kalır (oturum kopsa da)
RESUME = True             # varsa last.ckpt'den devam et

if MODE == "smoke":
    N_TOTAL, EPOCHS, BATCH = 40, 3, 4
else:
    N_TOTAL = 6000        # ~4800 train / 600 val / 600 test geometri
    EPOCHS = 150          # GPU süresine göre ayarlayın (epoch süresi eğitim çıktısında görünür)
    BATCH = 16            # GPU belleği yetmezse 8

HOLE_PROB = 0.3           # geometrilerin %30'u 1-2 delikli
EMBED_DIM = 128           # 128 × 4 katman ≈ 0.8M parametre
N_LAYERS = 4              # mass-aware attention blok sayısı
N_BASIS = 16              # Rayleigh–Ritz deneme uzayı boyutu (≥ 6 hedef mod)
LR = 2e-4
REFINE = 0                # 0 = saf model tahmini (FEM iyileştirme kapalı)
SEED = 0

In [ ]:
# ── 3. Kurulum ──────────────────────────────────────────────────────────
!apt-get -qq install -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1 > /dev/null
!pip -q install -r requirements.txt
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK — Runtime > Change runtime type > GPU")

In [ ]:
# ── 4. Yollar (Drive) ───────────────────────────────────────────────────
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = "/content/drive/MyDrive/rf_cavity"
else:
    WORK = "/content/rf_work"
TAG  = f"{N_TOTAL}_holes{HOLE_PROB}_seed{SEED}"
H5   = f"{WORK}/rf_cavity_{TAG}.h5"
PKL  = f"{WORK}/dataset_{TAG}_modes6.pkl"
LOGS = f"{WORK}/training_logs"
EXP  = f"eigenspace_{MODE}_{TAG}_d{EMBED_DIM}_L{N_LAYERS}"
OUT  = f"{WORK}/results_{EXP}"
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print("H5  :", H5); print("PKL :", PKL); print("EXP :", f"{LOGS}/{EXP}")

In [ ]:
# ── 5. Veri üretimi (gmsh + P2 FEM, 6 mod; varsa atlanır) ─────────────────
# full: 6000 geometri Colab'ın 2 CPU'sunda ~25-35 dk, ~1.7 GB H5.
if not os.path.exists(H5):
    !python src/data_gen/dataset_generator.py --mode random --n_total {N_TOTAL} --n_eigen_modes 6 --hole_prob {HOLE_PROB} --seed {SEED} --n_plot 5 --n_workers {os.cpu_count()} --h5_filename {H5} --plot_dir {WORK}/dataset_plots_{TAG}
else:
    print("Mevcut:", H5)

In [ ]:
# ── 6. H5 → PKL (13 özellik + ölçek + üçgenler, 6 mod; varsa atlanır) ─────
if not os.path.exists(PKL):
    !python convert.py --h5_filepath {H5} --output_path {PKL} --modes 0 1 2 3 4 5
else:
    print("Mevcut:", PKL)

In [ ]:
# ── 7. Veri özeti: topoloji ve yakın-dejenere oranları ──────────────────
import h5py, numpy as np, collections
with h5py.File(H5, "r") as f:
    keys = [k for k in f.keys() if "freqs" in f[k]]
    types = collections.Counter(str(f[k].attrs["shape_type"]) for k in keys)
    fr = np.array([f[k]["freqs"][:4] for k in keys])
gap = (np.diff(fr, axis=1) / fr[:, :-1]).min(axis=1)
print(f"{len(keys)} geometri | delikli: {sum(v for t, v in types.items() if '_hole' in t)} | tipler: {dict(types)}")
print(f"f1 aralığı: {fr[:,0].min():.2f}-{fr[:,0].max():.2f} GHz | ilk 4 modda göreli aralık <5%: {(gap < 0.05).mean():.1%}")

In [ ]:
# ── 8. Eğitim (EigenspaceOperator; kesilirse bu hücreyi tekrar çalıştırın → last.ckpt'den devam) ─
ckpt = f"{LOGS}/{EXP}/last.ckpt"
resume = f"--resume {ckpt}" if (RESUME and os.path.exists(ckpt)) else ""
print("Devam:", ckpt if resume else "yok (sıfırdan)")
!python train.py --config configs/eigenspace.yaml {resume} --override dataset.data_path={PKL} dataset.random_seed={SEED} model.embed_dim={EMBED_DIM} model.eigenspace.n_layers={N_LAYERS} model.n_basis={N_BASIS} training.learning_rate={LR} training.max_epochs={EPOCHS} training.batch_size={BATCH} training.num_workers=2 training.log_dir={LOGS} training.exp_name={EXP}

In [ ]:
# ── 9. Test değerlendirmesi (saf model; topoloji × özdeğer aralığı tablosu + CSV) ─
!python scripts/infer_val_all.py --checkpoint {LOGS}/{EXP} --data_path {PKL} --split test --csv {OUT}/test_metrics.csv --json {OUT}/test_metrics.json

In [ ]:
# ── 10. Görseller (GT | Tahmin | Hata), test split ──────────────────────
import glob
from IPython.display import Image, display
!python infer.py --checkpoint {LOGS}/{EXP} --data_path {PKL} --split test --num_samples 6 --output_dir {OUT}/plots --refine {REFINE}
for p in sorted(glob.glob(f"{OUT}/plots/*.png"))[:6]:
    print(os.path.basename(p)); display(Image(filename=p))

In [ ]:
# ── 11. (İsteğe bağlı) Eğitim eğrileri ──────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {LOGS}/{EXP}